In [1]:
"""
PRDA-04 - Customer Data Analysis (Istanbul Shopping Malls)
=============================================================
Dataset : Shopping transactions from 10 Istanbul malls (2021-2023)
Goal    :
    1. Shopping distribution by gender / age
    2. Which gender/age group generated more revenue & more sales?
    3. Category distribution relative to other columns
    4. Relationship between payment method and other variables
    5. Visualize and derive business insights/recommendations

Author  : Krishna Mohan Thirupathi
Tools   : Python, pandas, matplotlib/seaborn (mirrors Tableau/Power BI dashboard)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# ----------------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------------
DATA_PATH = "E:\Data Analyst Projects\customer project.csv"  # <-- update to your file location
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.head())
print(df.info())

df.columns = df.columns.str.strip().str.lower()
# Expected columns: invoice_no, customer_id, gender, age, category,
# quantity, price, payment_method, invoice_date, shopping_mall

# ----------------------------------------------------------------------
# 2. DATA CLEANING
# ----------------------------------------------------------------------
df = df.drop_duplicates()
df["invoice_date"] = pd.to_datetime(df["invoice_date"], dayfirst=True, errors="coerce")
df["gender"] = df["gender"].str.strip().str.title()
df["category"] = df["category"].str.strip().str.title()
df["payment_method"] = df["payment_method"].str.strip().str.title()

df["revenue"] = df["quantity"] * df["price"]

print("\nMissing values:\n", df.isna().sum())
print(f"\nTotal customers: {df['customer_id'].nunique()}")
print(f"Total transactions: {df['invoice_no'].nunique()}")
print(f"Total quantity sold: {df['quantity'].sum()}")

# ----------------------------------------------------------------------
# 3. SHOPPING DISTRIBUTION BY GENDER
# ----------------------------------------------------------------------
gender_counts = df["gender"].value_counts()
gender_qty = df.groupby("gender")["quantity"].sum()
gender_revenue = df.groupby("gender")["revenue"].sum()

print("\nTransaction count by gender:\n", gender_counts)
print("\nQuantity sold by gender:\n", gender_qty)
print("\nRevenue by gender:\n", gender_revenue)

top_gender_by_sales = gender_qty.idxmax()
top_gender_by_revenue = gender_revenue.idxmax()
print(f"\n-> Gender with more products sold: {top_gender_by_sales}")
print(f"-> Gender generating more revenue: {top_gender_by_revenue}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
gender_counts.plot(kind="pie", autopct="%1.1f%%", ax=axes[0], title="Shopping Distribution by Gender")
gender_qty.plot(kind="bar", ax=axes[1], title="Quantity Sold by Gender", color="#4C72B0")
gender_revenue.plot(kind="bar", ax=axes[2], title="Revenue by Gender", color="#DD8452")
plt.tight_layout()
plt.savefig("gender_distribution.png", dpi=150)
plt.close()

# Category distribution relative to gender
cat_gender = df.groupby(["category", "gender"])["revenue"].sum().unstack()
print("\nRevenue by category and gender:\n", cat_gender)

cat_gender.plot(kind="bar", figsize=(10, 5), title="Revenue by Category & Gender")
plt.ylabel("Revenue")
plt.tight_layout()
plt.savefig("category_by_gender.png", dpi=150)
plt.close()

# ----------------------------------------------------------------------
# 4. SHOPPING DISTRIBUTION BY AGE
# ----------------------------------------------------------------------
bins = [0, 20, 30, 40, 50, 60, 100]
labels = ["<20", "20-29", "30-39", "40-49", "50-59", "60+"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, right=False)

age_group_qty = df.groupby("age_group")["quantity"].sum()
age_group_revenue = df.groupby("age_group")["revenue"].sum()

print("\nQuantity sold by age group:\n", age_group_qty)
print("\nRevenue by age group:\n", age_group_revenue)

top_age_by_sales = age_group_qty.idxmax()
top_age_by_revenue = age_group_revenue.idxmax()
print(f"\n-> Age group with more products sold: {top_age_by_sales}")
print(f"-> Age group generating more revenue: {top_age_by_revenue}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
age_group_qty.plot(kind="bar", ax=axes[0], title="Quantity Sold by Age Group", color="#55A868")
age_group_revenue.plot(kind="bar", ax=axes[1], title="Revenue by Age Group", color="#C44E52")
plt.tight_layout()
plt.savefig("age_distribution.png", dpi=150)
plt.close()

# Category distribution relative to age group
cat_age = df.groupby(["category", "age_group"])["revenue"].sum().unstack()
plt.figure(figsize=(12, 6))
sns.heatmap(cat_age, annot=False, cmap="Blues")
plt.title("Revenue: Category vs Age Group")
plt.tight_layout()
plt.savefig("category_by_age_heatmap.png", dpi=150)
plt.close()

# ----------------------------------------------------------------------
# 5. PAYMENT METHOD ANALYSIS
# ----------------------------------------------------------------------
payment_dist = df["payment_method"].value_counts()
print("\nPayment method distribution:\n", payment_dist)

payment_by_gender = pd.crosstab(df["payment_method"], df["gender"])
payment_by_age = df.groupby("payment_method")["age"].mean()
payment_by_revenue = df.groupby("payment_method")["revenue"].sum().sort_values(ascending=False)

print("\nPayment method by gender:\n", payment_by_gender)
print("\nAverage age by payment method:\n", payment_by_age)
print("\nRevenue by payment method:\n", payment_by_revenue)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
payment_dist.plot(kind="pie", autopct="%1.1f%%", ax=axes[0], title="Payment Method Distribution")
payment_by_revenue.plot(kind="bar", ax=axes[1], title="Revenue by Payment Method", color="#8172B2")
plt.tight_layout()
plt.savefig("payment_method_analysis.png", dpi=150)
plt.close()

# Revenue by shopping mall (top performing malls)
mall_revenue = df.groupby("shopping_mall")["revenue"].sum().sort_values(ascending=False)
print("\nRevenue by shopping mall:\n", mall_revenue)

plt.figure(figsize=(8, 5))
mall_revenue.plot(kind="barh", color="#4C72B0")
plt.title("Revenue by Shopping Mall")
plt.xlabel("Revenue")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("revenue_by_mall.png", dpi=150)
plt.close()

# ----------------------------------------------------------------------
# 6. BUSINESS INSIGHTS & RECOMMENDATIONS
# ----------------------------------------------------------------------
print(f"""
Key Insights:
- {top_gender_by_revenue} customers contribute the most revenue overall.
- The {top_age_by_revenue} age group is the highest-spending segment --
  a strong target for loyalty programs and personalized offers.
- {payment_dist.index[0]} is the most-used payment method
  ({payment_dist.iloc[0]} transactions) -- ensure this channel has the
  smoothest checkout experience.
- {mall_revenue.index[0]} is the top-performing mall by revenue --
  consider replicating its product mix/promotions at lower-performing malls.

Recommendations:
- Run targeted marketing campaigns for the top revenue-generating
  gender and age segments.
- Offer incentives for under-used payment methods to diversify
  transaction risk and improve checkout conversion.
- Investigate why lower-performing malls lag and test best practices
  from top-performing malls.
""")

df.to_csv("istanbul_analysis_output.csv", index=False)
print("Saved cleaned/enriched dataset to istanbul_analysis_output.csv")


<>:26: SyntaxWarning: invalid escape sequence '\D'
<>:26: SyntaxWarning: invalid escape sequence '\D'
C:\Users\DELL\AppData\Local\Temp\ipykernel_23416\2780288887.py:26: SyntaxWarning: invalid escape sequence '\D'
  DATA_PATH = "E:\Data Analyst Projects\customer project.csv"  # <-- update to your file location


Shape: (99457, 10)
  invoice_no customer_id  gender  age    category  quantity    price  \
0    I100008     C199951    Male   65    Clothing         5  1500.40   
1    I100014     C138893    Male   55   Cosmetics         5   203.30   
2    I100015     C132779  Female   35    Clothing         2   600.16   
3    I100024     C244411  Female   67       Books         3    45.45   
4    I100027     C150002  Female   19  Technology         4  4200.00   

  payment_method invoice_date      shopping_mall  
0           Cash   10-07-2022  Emaar Square Mall  
1           Cash   18-06-2021     Viaport Outlet  
2     Debit Card   04-03-2021   Mall of Istanbul  
3    Credit Card   05-01-2023  Emaar Square Mall  
4           Cash   18-05-2022   Mall of Istanbul  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_no      99457 non-null  object

C:\Users\DELL\AppData\Local\Temp\ipykernel_23416\2780288887.py:94: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_group_qty = df.groupby("age_group")["quantity"].sum()
C:\Users\DELL\AppData\Local\Temp\ipykernel_23416\2780288887.py:95: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_group_revenue = df.groupby("age_group")["revenue"].sum()



Quantity sold by age group:
 age_group
<20      11296
20-29    57949
30-39    57875
40-49    57517
50-59    56922
60+      57153
Name: quantity, dtype: int64

Revenue by age group:
 age_group
<20       9051058.89
20-29    49044390.58
30-39    48288158.82
40-49    49234368.09
50-59    47593634.14
60+      48294183.73
Name: revenue, dtype: float64

-> Age group with more products sold: 20-29
-> Age group generating more revenue: 40-49


C:\Users\DELL\AppData\Local\Temp\ipykernel_23416\2780288887.py:113: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cat_age = df.groupby(["category", "age_group"])["revenue"].sum().unstack()



Payment method distribution:
 payment_method
Cash           44447
Credit Card    34931
Debit Card     20079
Name: count, dtype: int64

Payment method by gender:
 gender          Female   Male
payment_method               
Cash             26509  17938
Credit Card      21011  13920
Debit Card       11962   8117

Average age by payment method:
 payment_method
Cash           43.457421
Credit Card    43.427901
Debit Card     43.358534
Name: age, dtype: float64

Revenue by payment method:
 payment_method
Cash           1.128322e+08
Credit Card    8.807712e+07
Debit Card     5.059643e+07
Name: revenue, dtype: float64

Revenue by shopping mall:
 shopping_mall
Mall of Istanbul     50872481.68
Kanyon               50554231.10
Metrocity            37302787.33
Metropol AVM         25379913.19
Istinye Park         24618827.68
Zorlu Center         12901053.82
Cevahir AVM          12645138.20
Viaport Outlet       12521339.72
Emaar Square Mall    12406100.29
Forum Istanbul       12303921.24
Name: re